# Bag of Words (BoW) Pipeline

This notebook covers data loading, cleaning/preprocessing, Bag-of-Words feature
extraction, and model training/comparison for the fake news dataset (`title`, `text`, `label`).

Reusable functions from `utils.py` are used.

## 1. Setup

In [1]:
import os
os.chdir('../')
print(os.getcwd())

c:\Users\kriti\Dropbox\PC\Desktop\Ironhack_AI_Engineering\Projects\Project_2\Project-2-Natural-Language-Processing


In [2]:
# Core imports
import utils

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import hstack

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

## 2. Load Data

In [3]:
raw_data = pd.read_csv('dataset/data.csv')
print("Raw shape:", raw_data.shape)
# subject/date dropped here -- see EDA notebook for the leakage check behind this decision
raw_data = raw_data.drop(columns=['subject', 'date'])
print("Shape after dropping subject/date:", raw_data.shape)
raw_data.head()

Raw shape: (39942, 5)
Shape after dropping subject/date: (39942, 3)


,label,title,text
0,1,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...
1,1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...
2,1,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...
3,1,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...
4,1,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...


## 3. Train / Test Split

In [4]:
TEXT_COLUMNS = ['title', 'text']

X = raw_data.drop(columns='label')
y = raw_data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (31953, 2)
Test shape: (7989, 2)


## 4. Text Cleaning & Preprocessing

Uses `utils.preprocess_text_columns()`, which applies `utils.clean_and_preprocess_text()`
(HTML/URL/special-char cleaning + tokenize + stopword removal + lemmatization) to each
text column. Applied separately to train and test (fit-free, so no leakage).

This step is slow on the full dataset (tokenization + lemmatization over ~40k rows), so
results are cached to CSV — re-run the cell below only if you change the cleaning logic
or the underlying data. If you've already generated `X_train_clean.csv` / `X_test_clean.csv`
from another notebook (e.g. the TF-IDF one), you can skip straight to reloading.


In [6]:
# # Preprocess text columns and keep only preprocessed columns
# X_train_clean = utils.preprocess_text_columns(X_train, TEXT_COLUMNS)[[f'{c}_clean' for c in TEXT_COLUMNS]]
# X_test_clean = utils.preprocess_text_columns(X_test, TEXT_COLUMNS)[[f'{c}_clean' for c in TEXT_COLUMNS]]

# X_train_clean.columns = TEXT_COLUMNS
# X_test_clean.columns = TEXT_COLUMNS

# # Save splitted data
# X_train_clean.to_csv('X_train_clean.csv')
# X_test_clean.to_csv('X_test_clean.csv')

# X_train_clean.head()

### Reload cached cleaned text

In [7]:
X_train_clean = pd.read_csv('X_train_clean.csv', index_col=0)
X_test_clean = pd.read_csv('X_test_clean.csv', index_col=0)

print(X_train_clean.isna().sum())
print(X_test_clean.isna().sum())


title      5
text     549
dtype: int64
title      2
text     144
dtype: int64


### Fill null values with empty string

After cleaning/preprocessing, some rows end up empty (e.g. originally empty `text`,
or text that became empty after stripping HTML/special characters). Filling with `''`
keeps these rows usable for vectorization instead of dropping them.

In [8]:
X_train_clean = X_train_clean.fillna('')
X_test_clean = X_test_clean.fillna('')

print("Remaining NaNs:", X_train_clean.isna().sum().sum(), X_test_clean.isna().sum().sum())


Remaining NaNs: 0 0


## 5. Bag of Words Feature Extraction

One `CountVectorizer` per text column, fit on train only, then combined with `hstack`.

In [9]:
bow_vectorizers = {}
X_train_text_features = []
X_test_text_features = []

for col in TEXT_COLUMNS:
    vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 2))
    train_vec = vectorizer.fit_transform(X_train_clean[col])
    test_vec = vectorizer.transform(X_test_clean[col])

    bow_vectorizers[col] = vectorizer
    X_train_text_features.append(train_vec)
    X_test_text_features.append(test_vec)

    print(f"{col}: train shape {train_vec.shape}, test shape {test_vec.shape}")

title: train shape (31953, 5000), test shape (7989, 5000)
text: train shape (31953, 5000), test shape (7989, 5000)


In [10]:
# --- Combine all BoW features into one matrix ---
X_train_final = hstack(X_train_text_features)
X_test_final = hstack(X_test_text_features)

print("Final train feature matrix:", X_train_final.shape)
print("Final test feature matrix:", X_test_final.shape)

Final train feature matrix: (31953, 10000)
Final test feature matrix: (7989, 10000)


## 6. Model Training

Using `utils.train_evaluate_model()` — trains, evaluates (train/test accuracy, precision,
recall, F1, confusion matrix), prints a summary, saves the result to `results_log.jsonl`,
and pickles the fitted model to `models/`, all in one call.


In [11]:
# Container to collect results from every model
RESULTS = []

In [12]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_results = utils.train_evaluate_model(
    lr_model, 'BoW', 'Logistic Regression - BoW',
    X_train_final, y_train, X_test_final, y_test,
    comments="Baseline LR model with bow(5000 features, ngram_range=(1,2))",
    results_container=RESULTS
)

--- Logistic Regression - BoW | BoW (Baseline LR model with bow(5000 features, ngram_range=(1,2))) ---
Train Accuracy: 1.0000
Test Accuracy:  0.9966
Precision:      0.9967
Recall:         0.9965
F1 Score:       0.9966
Confusion Matrix:
 [[3976   13]
 [  14 3986]]
Model saved to: models\Logistic_Regression_-_BoW_BoW_2026-07-10_12-19-32.pkl



In [13]:
nb_model = MultinomialNB()
nb_results = utils.train_evaluate_model(
    nb_model, 'BoW', 'Naive Bayes - BoW',
    X_train_final, y_train, X_test_final, y_test,
    comments="Naive Bayes model with bow(5000 features, ngram_range=(1,2))",
    results_container=RESULTS
)


--- Naive Bayes - BoW | BoW (Naive Bayes model with bow(5000 features, ngram_range=(1,2))) ---
Train Accuracy: 0.9645
Test Accuracy:  0.9604
Precision:      0.9571
Recall:         0.9643
F1 Score:       0.9606
Confusion Matrix:
 [[3816  173]
 [ 143 3857]]
Model saved to: models\Naive_Bayes_-_BoW_BoW_2026-07-10_12-19-33.pkl



In [14]:
svm_model = LinearSVC()
svm_results = utils.train_evaluate_model(
    svm_model, 'BoW', 'SVM - BoW',
    X_train_final, y_train, X_test_final, y_test,
    comments="SVM model with bow(5000 features, ngram_range=(1,2))",
    results_container=RESULTS
)


--- SVM - BoW | BoW (SVM model with bow(5000 features, ngram_range=(1,2))) ---
Train Accuracy: 1.0000
Test Accuracy:  0.9962
Precision:      0.9958
Recall:         0.9968
F1 Score:       0.9963
Confusion Matrix:
 [[3972   17]
 [  13 3987]]
Model saved to: models\SVM_-_BoW_BoW_2026-07-10_12-19-33.pkl



In [15]:
rf_model = RandomForestClassifier(random_state=42)
rf_results = utils.train_evaluate_model(
    rf_model, 'BoW', 'Random Forest - BoW',
    X_train_final, y_train, X_test_final, y_test,
    comments="Random Forest model with bow(5000 features, ngram_range=(1,2))",
    results_container=RESULTS
) # 1m


--- Random Forest - BoW | BoW (Random Forest model with bow(5000 features, ngram_range=(1,2))) ---
Train Accuracy: 1.0000
Test Accuracy:  0.9986
Precision:      0.9980
Recall:         0.9992
F1 Score:       0.9986
Confusion Matrix:
 [[3981    8]
 [   3 3997]]
Model saved to: models\Random_Forest_-_BoW_BoW_2026-07-10_12-19-35.pkl



In [16]:
l

NameError: name 'l' is not defined

## 7. Model Comparison & Selection

In [ ]:
results_df = utils.load_results_from_file()
results_df = results_df.sort_values('f1_score', ascending=False).reset_index(drop=True)
results_df


In [ ]:
# --- Bar plot comparing metrics across models ---
metrics_to_plot = ['train_accuracy', 'test_accuracy', 'precision', 'recall', 'f1_score']
plot_df = results_df.melt(id_vars='model_name', value_vars=metrics_to_plot,
                           var_name='metric', value_name='score')

plt.figure(figsize=(12, 5))
sns.barplot(data=plot_df, x='model_name', y='score', hue='metric')
plt.title('Model Comparison Across Metrics (BoW)')
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# --- F1 score by model ---
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='model_name', y='f1_score')
plt.title('F1 Score by Model (BoW)')
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# --- Confusion matrices side by side (BoW models only) ---
bow_results = results_df[results_df['method'] == 'BoW'].reset_index(drop=True)

fig, axes = plt.subplots(1, len(bow_results), figsize=(5 * len(bow_results), 4))
if len(bow_results) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, bow_results.iterrows()):
    cm = np.array(row['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(row['model_name'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()


In [ ]:
# --- Best model selection ---
best_row = results_df.iloc[0]
print(f"Best model based on F1 Score: {best_row['model_name']} "
      f"(F1={best_row['f1_score']:.4f}, Test Accuracy={best_row['test_accuracy']:.4f})")
print(f"Saved model file: {best_row['model_path']}")


In [ ]:
# Reload the best model later, e.g. for inference:
# best_model = utils.load_model_from_file(best_row['model_path'])
